# 02 - MODEL TRAINING AND EVALUATION

Linear regression with gradient descent, implemented in pure Python.
Input: `data/processed/mobile_phone_price_clean.csv` (created by notebook 01).

In [ ]:
import pandas as pd
import random
import math

print("libraries imported successfully")

# LOAD THE CLEANED DATA

In [ ]:
df = pd.read_csv("../data/processed/mobile_phone_price_clean.csv")

df.head()

# ENCODE CATEGORICAL COLUMNS

In [ ]:
df=df.drop(columns=["Model"])
df_encoded = pd.get_dummies(df,columns=['Brand'],drop_first=True)
df_encoded.shape

# DEFINE FEATURES (X) AND TARGET (y)

In [ ]:
X= df_encoded.drop(columns=['Price'])

y = df_encoded["Price"]

print("Features (X):")
display(X.head())

print("\nTarget (y):")
display(y.tail())
print('Target shpe: ',y.shape)

# SPLIT DATA INTO TRAINING AND TESTING SETS

In [ ]:
def train_test_split(X,y,test_fac=0.20,seed=42):
    n=len(X)
    rng=random.Random(seed)
    indices=list(range(n))
    rng.shuffle(indices)

    n_train=int(n*(1-test_fac))
    train_idx,test_idx=indices[:n_train],indices[n_train:]

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    return(X_train, X_test, y_train, y_test)

X_train, X_test, y_train, y_test = train_test_split( X, y, test_fac=0.20,seed=42)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

# SCALE THE FEATURES

In [ ]:
def scaler(X_train, X_test):
    train_mean = X_train.mean(axis=0)
    train_std = X_train.std(axis=0, ddof=0).replace(0, 1e-9)  
    
    X_train_scaled = (X_train - train_mean) / train_std
    X_test_scaled = (X_test - train_mean) / train_std
    
    return X_train_scaled, X_test_scaled
    
X_train_scaled,X_test_scaled=scaler(X_train,X_test)
print("Features standardized successfully.")

# TRAIN A MACHINE-LEARNING MODEL

In [ ]:
def linear_model(X_train_scaled,y_train,lr=0.01, epochs=1000):
    n_samples=len(X_train_scaled)
    n_features=len(X_train_scaled[1])

    w=[0.0]*n_features
    b=0.0

    for epoch in range(epochs):
        #prediction
        y_pred=[]
        for i in range(n_samples):
            pred=0.0
            for j in range(n_features):
                pred+=X_train_scaled[i][j]*w[j]
            y_pred.append(pred+b)
            
        #error
        errors=[]
        for i in range(n_samples):
            error=y_pred[i]-y_train[i]
            errors.append(error)
                
        #loss
        dw=[0.0] * n_features
        for i in range(n_features):
            for j in range(n_samples):
                dw[i]+=(2 / n_samples) * errors[j] * X_train_scaled[j][i]
                                              
        db = (2 / n_samples) * sum(errors)

        #update weight and bais
        for j in range(n_features):
            w[j] -= lr * dw[j]
        b -= lr * db

        if (epoch + 1) % 100 == 0:
            mse = sum(err ** 2 for err in errors) / n_samples
            print(f"Epoch {epoch + 1}/{epochs} - Loss (MSE): {mse:.2f}")
    final_mse=sum(err ** 2 for err in errors) / n_samples
    return w, b, final_mse

def predict(X, w, b):
    n_samples=len(X)
    n_features=len(w)
    predictions=[]
    for i in range(n_samples):
        pred = sum(X[i][j] * w[j] for j in range(n_features)) + b
        predictions.append(pred)
    return predictions

    
X_train_list = X_train_scaled.values.tolist()
y_train_list = y_train.values.tolist()
#y_train_log_list = y_train_log.values.tolist()

w, b,MSE = linear_model(X_train_list, y_train_list, lr=0.01, epochs=15000)


print(f'\nweight: {w}')
print(f"learned_bais: {b:.2f}")
print(f"Loss: {MSE:.2f}")

# EVALUATE THE MODEL

In [ ]:
def evaluate_model(X_test, y_test, w, b,):

    y_pred = predict(X_test, w, b)
    y_actual = y_test

    n_samples = len(y_actual)
    mae = sum(abs(y_actual[i] - y_pred[i]) for i in range(n_samples)) / n_samples
    mse = sum((y_actual[i] - y_pred[i]) ** 2 for i in range(n_samples)) / n_samples
    rmse = math.sqrt(mse)
    
    mean_actual = sum(y_actual) / n_samples
    ss_tot = sum((y_actual[i] - mean_actual) ** 2 for i in range(n_samples))
    ss_res = sum((y_actual[i] - y_pred[i]) ** 2 for i in range(n_samples))
    r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0.0

    print("=" * 40)
    print("        MODEL EVALUATION METRICS        ")
    print("=" * 40)
    print(f"Mean Absolute Error (MAE)     : {mae:.2f}")
    print(f"Mean Squared Error (MSE)     : {mse:.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
    print(f"R-squared Score (R²)          : {r2:.4f}")
    print("-" * 40)
    
    print("\nFirst 5 Actual vs. Predicted Values:")
    for i in range(min(5, n_samples)):
        print(f"Actual: {y_actual[i]:.2f} | Predicted: {y_pred[i]:.2f}")

    return {"mae": mae, "mse": mse, "rmse": rmse, "r2": r2}

    
X_test_list = X_test_scaled.values.tolist()
y_test_list=y_test.values.tolist()
metrics = evaluate_model(X_test_list, y_test_list, w, b)